### Harry potter rag 

In [1]:
from dotenv import load_dotenv
import os
import pandas as pd

load_dotenv()
api_key = os.getenv("GEMINI_API_KEY") # This is the API key for the Gemini API, which is used to access the language model.

csv_path = "data/harry_potter_cleaned.csv" 

df = pd.read_csv(csv_path)
df.head() 

,name,gender,job,house,patronus,species,blood_status,loyalty,skills,birth,...,titles,category,hand,light,difficulty,inventors,ingredients,side_effects,time,title
0,Albus Percival Wulfric Brian Dumbledore,Male,Headmaster,Gryffindor,Phoenix,Human,Half-blood,Dumbledore's Army | Order of the Phoenix | Hog...,Considered by many to be one of the most power...,Late August 1881,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Quirinus Quirrell,Male,Defence Against the Dark Arts(1991-1992),Ravenclaw,Non-corporeal,Human,Half-blood,Lord Voldemort,Learned in the theory of Defensive Magic| less...,"26 September,1970 or earlier",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Fred Weasley,Male,Student,Gryffindor,Unknown,Human,Pure-blood,Dumbledore's Army | Order of the Phoenix | Hog...,Beater,"1 April, 1978",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Nymphadora Tonks,Female,Auror,Hufflepuff,"Jack rabbit, Wolf",Human,Half-blood,Ministry of Magic | Order of the Phoenix,"Talented Auror, Metamorphmagus",1 September 1972- 31 August 1973,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Lavender Brown,Female,Student,Gryffindor,Unknown,Human,Pure-blood,Dumbledore's Army |Hogwarts School of Witchcra...,Divination,1 September 1979- 31 August 1980,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [2]:
# Load the CSV file using the CSVLoader from langchain_community
from langchain_community.document_loaders.csv_loader import CSVLoader

loader = CSVLoader(file_path=csv_path)
documents = loader.load()
print(f"Number of documents: {len(documents)}")
print(f"First document: {documents[0]}")

/Users/opheliathomasson/Documents/TUC_datamanager/datakvalité/kunskapskontroll/data_kvalit-_kunskapskontroll/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Number of documents: 5766
First document: page_content='name: Albus Percival Wulfric Brian Dumbledore
gender: Male
job: Headmaster
house: Gryffindor
patronus: Phoenix
species: Human
blood_status: Half-blood
loyalty: Dumbledore's Army | Order of the Phoenix | Hogwarts School of Witchcraft and Wizardry
skills: Considered by many to be one of the most powerful wizards of his time
birth: Late August 1881
death: 30 June, 1997
source: characters
incantation: 
type: 
effect: 
characteristics: 
alias_names: 
titles: 
category: 
hand: 
light: 
difficulty: 
inventors: 
ingredients: 
side_effects: 
time: 
title: ' metadata={'source': 'data/harry_potter_cleaned.csv', 'row': 0}


In [3]:
# Creating a embeddings model using the GoogleGenerativeAIEmbeddings class from langchain_google_genai
#from langchain_google_genai import GoogleGenerativeAIEmbeddings
#embeddings = GoogleGenerativeAIEmbeddings(
#    model="models/gemini-embedding-001",
#    google_api_key=api_key
#)


In [4]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/var/folders/p5/jkbyc74n1jj4grxpk0w1ltf80000gn/T/ipykernel_85782/2671871813.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3664.95it/s]


In [5]:
# Create a Chroma vector store and add the documents to it
from langchain_chroma import Chroma

vector_store = Chroma(
    collection_name="harry_potter_hf",
    embedding_function=embeddings,
    persist_directory="./chroma_harry_potter_hf_db",
)

In [6]:
# Add a smaller sample first before adding all documents, to make sure everything works as expected. 
# And to avoid hitting rate limits with the embedding model.
#sample_documents = documents[:500]

#document_ids = vector_store.add_documents(documents=sample_documents)

#print(len(document_ids))

In [7]:
batch_size = 500

for i in range(0, len(documents), batch_size):
    batch = documents[i:i + batch_size]

    vector_store.add_documents(documents=batch)

    print(f"Added documents {i} to {i + len(batch)}")

Added documents 0 to 500
Added documents 500 to 1000
Added documents 1000 to 1500
Added documents 1500 to 2000
Added documents 2000 to 2500
Added documents 2500 to 3000
Added documents 3000 to 3500
Added documents 3500 to 4000
Added documents 4000 to 4500
Added documents 4500 to 5000
Added documents 5000 to 5500
Added documents 5500 to 5766


In [8]:
# Checking the similarity search with a sample question.
question = "What house does Harry Potter belong to?"

results = vector_store.similarity_search(question, k=3)

for result in results:
    print(result.page_content)
    print("---")

name: Henry Potter
gender: Male
job: ['Member of the Wizengamot (1913–1921)']
house: 
patronus: 
species: Human
blood_status: Pure-blood
loyalty: 
skills: 
birth: 
death: 
source: master
incantation: 
type: 
effect: 
characteristics: 
alias_names: ['Harry (by family and friends)']
titles: []
category: 
hand: 
light: 
difficulty: 
inventors: 
ingredients: 
side_effects: 
time: 
title: 
---
name: Kreacher
gender: Male
job: ["House of Black's house-elf (in/pre 1979–1996)", "Harry Potter's house-elf (1996–?)", 'Hogwarts kitchen worker (1996–?)']
house: 
patronus: 
species: House-elf
blood_status: 
loyalty: 
skills: 
birth: 
death: 
source: master
incantation: 
type: 
effect: 
characteristics: 
alias_names: []
titles: []
category: 
hand: 
light: 
difficulty: 
inventors: 
ingredients: 
side_effects: 
time: 
title: 
---
name: Harry James Potter
gender: Male
job: Student
house: Gryffindor
patronus: Stag
species: Human
blood_status: Half-blood
loyalty: Albus Dumbledore | Dumbledore's Army | Ord

In [9]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(
    model="gemini-3-flash-preview",
    google_api_key=api_key
)

In [10]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

In [11]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

template = """
You are a helpful assistant that answers questions about a Harry Potter dataset.
Use the retrieved data to answer the question.
If the retrieved data does not contain the information needed to answer, say you don't know.

Question: {question}

Retrieved data:
{retrieved_data}
"""


In [12]:
prompt = PromptTemplate.from_template(template)
parser = StrOutputParser()

In [13]:
chain = (
    {
        "retrieved_data": retriever,
        "question": RunnablePassthrough()
    }
    | prompt
    | model
    | parser
)

In [14]:
question = input("Ask a question about the Harry Potter dataset: ")


In [15]:
answer = chain.invoke(question)

In [16]:
print(answer)

Based on the retrieved data, there are two potions that cause a person to become infatuated:

*   **Love potion:** This pink-coloured potion causes infatuation with whoever offered it. Its side effects include embarrassment on the part of the drinker. 
*   **First Love Beguiling Bubbles:** This red-coloured potion, invented by Fred and George Weasley, causes the drinker to become infatuated with the person who gave them the potion.
